# AML intensive treatment infection prediction model -- Data preprocessing

data used (without imputation)
- data_final1_wo_imputation.csv 

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

pd.set_option('display.max_columns', None)

Data preprocessing for all of these

In [ ]:
interpolation = pd.read_csv('/path/to/data_final1_wo_interpolation.csv')
uncertainity = pd.read_csv('/path/to/data_final1_uncertainty.csv')
NOimputation = pd.read_csv('/path/to/data_final1_wo_imputation.csv')

## DATA

In [ ]:
# remove these columns from the data
delete = pd.read_excel('/path/to/data_final_dg_documentation0.xlsx')
delist = delete[delete['poista_analyyseista']==1]['names'][2:].tolist()
delist.remove('treatment_date')
delist.remove('naytteenotto_hetki')
delist.append('time_from_dg')
delist

In [6]:
# differently imputed data to try
dflist = [interpolation, uncertainity, NOimputation]
dfnames = ['interpolation', 'uncertainty', 'NOimputation']

remove treatment cycles from NOimputation that have over 85% NAns in specific columns

In [ ]:
cols = ["temperature", "b_neut"]   # whatever columns you care about
threshold = 0.85

bad_groups = set()   # will store (henkilotunnus, cycle_number) pairs

for col in cols:
    frac_nan_per_group = (
        NOimputation.groupby(["henkilotunnus", "cycle_number"])[col]
                    .apply(lambda s: s.isna().mean())
    )

    # groups where this column has ≥ 85% NaNs
    bad_here = frac_nan_per_group[frac_nan_per_group >= threshold]

    print(f"\nColumn: {col}")
    print("Number of groups to drop:", len(bad_here))

    # add these groups to the global set
    bad_groups.update(bad_here.index)

BAD_GROUPS = list(bad_groups)

## Function

In [9]:

def process_data(df, dfname, delist):
    '''  Processes the dataframe for the infection prediction model  '''
    
    print(f"-------------- {dfname} --------------")
    inf_cycles_num = df.groupby(['henkilotunnus', 'cycle_number']).count()
    print("Cycles before preprocessing:", len(inf_cycles_num))

    # create countdown col from the starting day of the treatment
    #df['countdown'] = df.groupby(['henkilotunnus', 'cycle_number']).cumcount()
    # make sure it's datetime
    df["naytteenotto_hetki"] = pd.to_datetime(df["naytteenotto_hetki"])

    g = df.groupby(["henkilotunnus", "cycle_number"])["naytteenotto_hetki"]

    # 0-based day offset from the earliest timestamp in the group
    df["countdown"] = (df["naytteenotto_hetki"] - g.transform("min")).dt.days
    #df[['henkilotunnus', 'naytteenotto_hetki', 'infektion_binary', 'cycle_number', 'countdown_days']].head(20)

    # delete unneccessary columns
    print('Cols before del:', len(df.columns))
    df = df.drop(columns=delist, errors='ignore')
    print('Cols after del:', len(df.columns))
    print()
    
    # categorical data to int
    categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
    categorical_cols.remove('henkilotunnus')
    categorical_cols.remove('treatment_date')

    #categorical_cols.remove('naytteenotto_hetki')
    categorical_cols = list(set(categorical_cols) - set(delist))
    categorical = ['ELN', 'sukupuoli_selite', 'sykli']
    
    # categorical data to dymmy
    df = pd.get_dummies(df, columns = categorical, dtype=float) 
    df = df.drop(columns=['sykli_KONS', 'sukupuoli_selite_Nainen'])

    # replace true-false with 0 and 1
    non_cat = list(set(categorical_cols) - set(categorical))
    if 'naytteenotto_hetki' in non_cat:
        non_cat.remove('naytteenotto_hetki')
    df[non_cat] = df[non_cat].apply(pd.to_numeric, errors='coerce')
    
    # Check for infinity values
    is_infinite = df.isin([np.inf, -np.inf])
    res = is_infinite.any().any()
    print('Does df contain infinite values:', res)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)

    print(df.groupby(["henkilotunnus", "cycle_number"]).ngroups)
    
    # mark groups that should be dropped
    drop_group = (
        df.groupby(["henkilotunnus", "cycle_number"])
            .apply(lambda g: ((g["countdown"] < 6) & (g["fn_day"] == 1)).any())
    )

    # keep only groups that are NOT marked for dropping
    df = df[~df.set_index(["henkilotunnus", "cycle_number"]).index.isin(drop_group[drop_group].index)]
    print(df.groupby(["henkilotunnus", "cycle_number"]).ngroups)
    
    # -------------------------- Drop bad groups -------------------------
    print("--> groups that have over 85% of nan values (in b_neut or temperature) dropped:", len(BAD_GROUPS))
    # Build a boolean mask: True if this row belongs to a bad group
    mask_bad = df[["henkilotunnus", "cycle_number"]] \
        .apply(tuple, axis=1) \
        .isin(BAD_GROUPS)

    # Keep only rows that are NOT in bad groups
    df = df[~mask_bad]
    
    # --------------------------------------------------------------------

    df['transfusion_dependent_6x'] = df['transfusion_dependent_6x'].astype(float)
    

    # lastly lets only keep adult patients

    # BEFORE filtering, save the original unique IDs
    orig_ids = set(df['henkilotunnus'].unique())
    
    # Coerce to numeric just in case
    df['age'] = pd.to_numeric(df['age'], errors='coerce')
    
    # Keep only IDs whose minimum age across all their rows is >= 18
    valid_mask = df.groupby('henkilotunnus')['age'].transform('min') >= 18
    df = df[valid_mask].copy()

    # ... after your filtering step to keep adults only ...
    kept_ids = set(df['henkilotunnus'].unique())
    
    discarded_count = len(orig_ids - kept_ids)
    print('--> Discarded pediatric patients:', discarded_count)

    # statistics
    print()
    inf_cycles = df.groupby(['henkilotunnus', 'cycle_number'])['infektion_binary'].sum().gt(0).mean() * 100
    inf_cycles_num = df.groupby(['henkilotunnus', 'cycle_number']).count()
    print("Number of patients:", len(df['henkilotunnus'].drop_duplicates()))
    print("Number of cycles:", len(inf_cycles_num))
    print(f"Infection cycles (%): {inf_cycles}\n")
    pr = (len(df[df['infektion_binary']==1]) / len(df[df['infektion_binary']==0]) )
    print("Amount of infections:", pr)
    print("Number of columns:", len(df.columns))
    print('number of rows', len(df))
    
    #print("\n-----------------------------------------\n")
    print('\n')

    return df

In [ ]:
df_dict = {}
for i, (df, name) in enumerate(zip(dflist, dfnames)):
    df_dict[name] = process_data(df, name, delist)

In [11]:
remove_list = pd.read_csv("/path/to/exclude_vars.csv")['variables'].tolist()

In [12]:
for name in df_dict: 
    rmv_list = list(set(remove_list) & set(df_dict[name].columns.tolist()))
    df_dict[name].drop(columns=rmv_list, inplace=True)
    df_dict[name].to_csv(f"/path/to/infection_model/data/model_data_{name}.csv")
    len(df_dict[name].columns.tolist())


In [13]:
df_dict['NOimputation'][['henkilotunnus', 'cycle_number']].drop_duplicates().to_csv("/path/to/infection_model/data/model_treatment_cycles.csv")